# 9.6 Specifying Data Using Polars DataFrames

Polars is a DataFrame library for Python designed around a lazy query engine and a columnar, Apache Arrow-based memory model. Compared to Pandas, it consistently achieves substantially higher throughput on large datasets and benefits from automatic query optimisation when operations are chained. These advantages make Polars attractive in data-intensive applications where preprocessing—joining, filtering, aggregating—is a significant part of the overall workflow.

Since amplpy expects data in the form of Pandas DataFrames, a typical workflow combines the two libraries: Polars is used for all data loading and transformation, and the result is converted to Pandas with the `.to_pandas()` method immediately before the call to `set_data()` or the parameter assignment. This division of labour—Polars for performance, Pandas as the interface to AMPL—requires only a single extra method call and adds no meaningful overhead at the scale of typical optimisation datasets.

In [1]:
# install dependencies
%pip install -q amplpy pandas numpy polars

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Note: you may need to restart the kernel to use updated packages.
Licensed to AMPL Community Edition License for <uykun96@gmail.com>.


We use the same diet model as in previous sections.

In [2]:
%%writefile diet.mod

set NUTR;
set FOOD;
set LINKS within (NUTR cross FOOD);

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];
param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];
param amt {NUTR,FOOD} >= 0;

Overwriting diet.mod


In [3]:
ampl.read("diet.mod")

## 9.6.1 Constructing and Converting Polars DataFrames

The data definition follows the same logical structure as in Section 9.4. We first build Polars DataFrames for `food_df` and `nutr_df`, then call `.to_pandas().set_index(...)` to convert each one to the indexed Pandas form that `set_data()` expects. The conversion is done inline so that the Polars DataFrame is never stored in a separate variable; only the Pandas version, which is needed by AMPL, is retained.

In [4]:
import polars as pl

# ---------------------------
# Food data (cost and bounds)
# ---------------------------
food_df = pl.DataFrame({
    "FOOD": ["BEEF", "CHK", "FISH", "HAM", "MCH", "MTL", "SPG", "TUR"],
    "cost": [3.59, 2.59, 2.29, 2.89, 1.89, 1.99, 1.99, 2.49],
    "f_min": [2] * 8,   # Minimum allowed consumption
    "f_max": [10] * 8   # Maximum allowed consumption
}).to_pandas().set_index("FOOD")

# ---------------------------
# Nutrient data (bounds)
# ---------------------------
nutr_df = pl.DataFrame({
    "NUTR": ["A", "C", "B1", "B2", "NA", "CAL"],
    "n_min": [700, 700, 700, 700, 0, 16000],
    "n_max": [20000, 20000, 20000, 20000, 50000, 24000]
}).to_pandas().set_index("NUTR")

# ---------------------------
# Nutrient-content matrix
# ---------------------------
amt_df = pl.DataFrame({
    "FOOD": ["BEEF", "CHK", "FISH", "HAM", "MCH", "MTL", "SPG", "TUR"],
    "A":   [60, 8, 8, 40, 15, 70, 25, 60],
    "C":   [20, 0, 10, 40, 35, 30, 50, 20],
    "B1":  [10, 20, 15, 35, 15, 15, 25, 15],
    "B2":  [15, 20, 10, 10, 15, 15, 15, 10],
    "NA":  [928, 2180, 945, 278, 1182, 896, 1329, 1397],
    "CAL": [295, 770, 440, 430, 315, 400, 379, 450]
}).to_pandas().set_index("FOOD")

Once the DataFrames are in their Pandas form the loading calls are identical to those in Section 9.4:

In [5]:
# Load structured data into AMPL
ampl.set_data(food_df, "FOOD")   # Defines FOOD set and parameters: cost, f_min, f_max
ampl.set_data(nutr_df, "NUTR")   # Defines NUTR set and parameters: n_min, n_max

This workflow—Polars for construction, `.to_pandas()` at the boundary, `set_data()` into AMPL—is a clean pattern that scales gracefully. In larger applications where the data originates from Parquet files, remote databases, or multi-step ETL pipelines, the Polars part of the pipeline can be arbitrarily complex while the AMPL-loading code remains a fixed two- or three-line block. The conversion overhead is negligible compared to the cost of preprocessing large datasets in pure Pandas.